# Campaign · Milestone 3 — **TTM-r2** on every non-C-MAPSS dataset

One of five sibling notebooks (`chronos` / `moment` / `timesfm` / `ttm` / `moirai`), **one backbone per Colab runtime** — the five stacks are mutually incompatible (`requirements/README.md`) and must never share an environment. Each notebook runs the full dataset scale-up for its ONE frozen backbone: embed → cache to Drive → data-scaling sweep (+ the shared baselines) → horizon eval → figures, dataset by dataset. C-MAPSS (FD001–FD004) is **not** here — it completed in `milestone_1/` + `milestone_2/`.

`run_campaign` routes each dataset to the arm its physics supports (CHANGES.md §54–§56):

| dataset(s) | arm | headline CSV (in `results/<dataset>/`) |
|---|---|---|
| XJTU-SY · DS01–DS08c · DSALL | RUL data-scaling sweep + horizon | `<ds>_granite-timeseries-ttm-r2_results_v2.csv`, `<ds>_granite-timeseries-ttm-r2_horizon.csv` |
| MetroPT-3 · Backblaze (censored fleets) | binary **alarm** sweep | `<ds>_granite-timeseries-ttm-r2_alarm_results.csv` |
| Hydraulic (no failure events at all) | **RQ-F taxonomy probe** | `Hydraulic_granite-timeseries-ttm-r2_taxonomy.csv` |

**Results layout:** every artifact lands in `results/<dataset>/…` on Drive — one folder per dataset, figures in `results/<dataset>/figures/`. Filenames keep the `<dataset>_<model>_` prefix, so cross-model scoring can still glob `results/*/*_results_v2.csv`.

**Before running:** `Runtime ▸ Change runtime type ▸ GPU`; use a **fresh runtime** (restart if another backbone ran here); run top-to-bottom. Every stage is **restartable** — re-running skips cached embeddings and completed sweep cells, so it is safe (and expected) to finish the big datasets (N-CMAPSS, DSALL, Backblaze) across several sessions. Datasets not downloaded are skipped with a printed notice, never an error.


In [ ]:
# 1) Setup — clone the code fresh from GitHub (re-run-safe); Drive never mirrors the repo.
import os, sys, subprocess

REPO_URL    = 'https://github.com/blozanod/Predictive-Maintenance-LSTM.git'
REPO_BRANCH = 'main'                       # branch to run (a public repo needs no token)
CLONE_DIR   = '/content/Predictive-Maintenance-LSTM'
if not os.path.isdir(os.path.join(CLONE_DIR, '.git')):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH,
                    REPO_URL, CLONE_DIR], check=True)
else:
    subprocess.run(['git', '-C', CLONE_DIR, 'fetch', '--depth', '1', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', CLONE_DIR, 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', CLONE_DIR, 'reset', '--hard', f'origin/{REPO_BRANCH}'], check=True)
if CLONE_DIR not in sys.path:
    sys.path.insert(0, CLONE_DIR)          # import src.* from the fresh clone
os.chdir(CLONE_DIR)
print('code at', CLONE_DIR, '| branch', REPO_BRANCH)


In [ ]:
# 2) install ONLY TTM-r2's isolated stack (see requirements/ for the pin rationale;
#    the flags mirror notebooks/campaign/milestone_1/ttm.ipynb exactly — §59).
#    This install CHANGES torch/torchvision. After it finishes, do
#    Runtime ▸ Restart session, then RE-RUN FROM THE TOP — the clone cell is
#    re-run-safe and the install is cached, so the second pass is quick. (Skipping
#    the restart shows up as `operator torchvision::nms does not exist` at the
#    import-assert cell below — that error means "restart now".)
!pip install -r requirements/ttm.txt


### Top-up: the head / baseline / loader deps the backbone stack doesn't carry

`requirements/<model>.txt` is an **embedding-only** stack (CHANGES.md §48). This notebook
trains heads and baselines and parses the raw datasets in the SAME runtime, so it needs:

* **`coral-pytorch`** — the CORN ordinal loss arm (`--no-deps` is load-bearing: it keeps
  this stack's pinned torch/torchvision untouched).
* **`lightgbm`** — the `gbm` / `gbm_age` / alarm-GBM baselines.
* **`pycatch22`** — the `catch22_gbm` indicator foil + the RQ-F taxonomy feature source.
* **`h5py`** (N-CMAPSS `.h5`) and **`pyarrow`** (Backblaze daily CSVs) — usually already
  in Colab's base image; the install is a no-op then.

`minirocket` (sktime + numba) is deliberately **absent**: its numpy pins fight the
backbone stacks (the §48 precedent) — see the baseline-roster note at the campaign cell.

The cell asserts every import — **including the backbone itself** — so a failed install
dies HERE, not after the first dataset's multi-minute parse (§59: the first MOMENT run
lost a session exactly that way). If `cuda` prints `False`, stop: embedding would run
on CPU. On TTM/Moirai a `torchvision::nms` error here means Runtime ▸ Restart session,
then re-run from the top.


In [ ]:
# 2b) top-up deps — see the note above. --no-deps keeps the pinned torch untouched (§48).
!pip install --no-deps coral-pytorch
!pip install lightgbm pycatch22 h5py pyarrow

import tsfm_public                           # the backbone itself — fail HERE, not mid-run (§59)
from coral_pytorch.losses import corn_loss   # CORN loss arm
import lightgbm, pycatch22, h5py, pyarrow    # baselines + dataset parsers
import torch
assert torch.cuda.is_available(), 'no CUDA — embedding would silently run on CPU'
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '| tsfm_public OK | deps OK')


In [ ]:
# 3) mount Google Drive — it holds Data/ (raw datasets), cache/ (embeddings) and results/.
from google.colab import drive
drive.mount('/content/drive')


## Where the data lives

`DATA_ROOT` must hold the raw datasets, one folder each (loaders accept the shipped
names and tolerate one nesting level — CHANGES.md §26):

```
<DATA_ROOT>/
  XJTU-SY/     the 3 condition folders (35Hz12kN/ 37.5Hz11kN/ 40Hz10kN/)
  N-CMAPSS/    the DS01…DS08c .h5 files, flat
  MetroPT-3/   MetroPT3(AirCompressor).csv       (MetroPT3.csv also accepted)
  Hydraulic/   the 17 sensor .txt files + profile.txt
  Backblaze/   the daily YYYY-MM-DD.csv files, any nesting
               (fetch with notebooks/backblaze_download.ipynb)
```

The default is `pdm_tsfm/Data` — the SAME Drive folder Milestones 1–2 used for `cache/`
and `results/`, so everything lives in one place. **The retired `phase_b.ipynb` defaulted
to a DIFFERENT folder** (`MyDrive/Predictive Maintenance LSTM/Data`); if your downloads
live there, either move them into `pdm_tsfm/Data` or point `DATA_ROOT` at that folder.
A dataset that is missing is skipped with a notice — the preflight cell below tells you
exactly what was found before anything runs.


In [ ]:
# 4) the CANONICAL config — the §12 winner shape for every cache-key field. Per-dataset
#    protocol (window / max_rul / alarm_horizon / …) comes from
#    campaign.DEFAULT_DATASET_OVERRIDES — inspect it below, don't hand-edit it here.
from src.config import Config

DRIVE     = '/content/drive/MyDrive/pdm_tsfm'  # cache/ + results/ — SAME folder as Milestones 1–2
DATA_ROOT = f'{DRIVE}/Data'                    # ← the folder holding the raw datasets (see above)
MODEL     = 'ibm-granite/granite-timeseries-ttm-r2'
TAG       = 'ttm'                # short session suffix for the probe CSVs
MODEL_TAG = MODEL.split('/')[-1]               # the tag campaign filenames carry
RESULTS   = f'{DRIVE}/results'                 # per-dataset subfolders are created below

config = Config(
    data_root=DATA_ROOT,
    cache_dir=f'{DRIVE}/cache',    # ONE flat cache — keys already encode dataset + model
    results_dir=RESULTS,           # per-dataset runs override this to results/<dataset>/
    model_name=MODEL,
    tsfm_context_length=256,       # recorded FD001 winner (CHANGES.md §12)
    pooling='mean',
    head_features='emb+locscale',  # head knob; not a cache key (CHANGES.md §9)
    # embed_batch_size=64,         # lower on a T4 (esp. N-CMAPSS: 37 channels; DSALL is the giant)
)

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device


## Preflight — what is on Drive, and what protocol each dataset gets

Read this before you read any number: it states, per dataset, what a "cycle" and a
"unit" mean, whether the dataset is censored (→ alarm arm) or classification-only
(→ RQ-F probe), and what the alarm horizon is measured in. Missing datasets are listed
here loudly — the retired `phase_b.ipynb` run produced no results precisely because
every dataset was silently absent at its default path.


In [ ]:
# The full non-C-MAPSS roster, cheapest families first. Trim this list to slice work
# across sessions — every stage is restartable and skips finished work.
DATASETS = ['XJTU-SY', 'Hydraulic', 'MetroPT-3',
            'DS01', 'DS02', 'DS03', 'DS04', 'DS05', 'DS06', 'DS07', 'DS08a', 'DS08c',
            'DSALL', 'Backblaze']

from src import datasets as datasets_mod
from src.campaign import DEFAULT_DATASET_OVERRIDES, RUL_ONLY_STAGES, TIME_TO_EVENT_STAGES

available, missing = [], []
for ds in DATASETS:
    over = {k: v for k, v in DEFAULT_DATASET_OVERRIDES.get(ds, {}).items()
            if k != 'sensor_columns'}
    probe = config.replace(dataset=ds, sensor_columns=None, **over)
    if datasets_mod.is_available(probe):
        available.append(ds)
        print(f'FOUND   {ds:10s} kind={probe.dataset_kind():10s} '
              f'censored={probe.is_censored_dataset()!s:5s} '
              f'classification={probe.is_classification_dataset()!s:5s} '
              f'channels={probe.num_channels():3d} alarm_horizon={probe.alarm_horizon}')
        if over:
            print(f'        {"":10s} overrides: {over}')
    else:
        missing.append(ds)
        print(f'missing {ds:10s} — not found under {DATA_ROOT} (will be skipped)')

print(f'\n{len(available)} dataset(s) ready, {len(missing)} missing.')
print('RUL-only stages skipped on censored fleets:', RUL_ONLY_STAGES)
print('time-to-event stages skipped on Hydraulic :', TIME_TO_EVENT_STAGES)


## Campaign — every dataset, this one backbone

One `run_campaign` call per dataset so each writes into its own `results/<dataset>/`
folder. A failure in one dataset never kills the loop; the summary at the end says per
dataset whether it ran, was skipped (no data), or failed (with the error).

**Baseline roster** (`DECISION`, CHANGES.md §58): `predict_mean · gbm · cnn · lstm ·
catch22_gbm`. `minirocket` is dropped on backbone runtimes — its sktime/numba numpy pins
fight the backbone stacks (the §48 precedent), and in every full-fleet C-MAPSS cell the
strongest baseline was gbm/gbm_age, never minirocket. `catch22_gbm` joins as the
hand-crafted-indicator foil (RQ-D). Censored fleets use the alarm sweep's own roster
(`alarm_base_rate · alarm_gbm`) automatically.


In [ ]:
import traceback
from src.campaign import run_campaign

STAGES    = ['cache', 'sweep', 'fairness', 'horizon', 'figures']
BASELINES = ['predict_mean', 'gbm', 'cnn', 'lstm', 'catch22_gbm']   # see roster note above

summary = []
for ds in DATASETS:
    cfg = config.replace(results_dir=f'{RESULTS}/{ds}')   # ← the per-dataset results folder
    try:
        summary += run_campaign(cfg, datasets=[ds], models=[MODEL], stages=STAGES,
                                baseline_names=BASELINES, device=device)
    except Exception as e:                 # keep the run-all alive across datasets
        traceback.print_exc()
        summary.append({'dataset': ds, 'model': MODEL, 'status': 'failed',
                        'error': f'{type(e).__name__}: {e}'})

print('\n=== session summary ===')
for row in summary:
    files = sorted(k for k in row if k.endswith('_csv'))
    print(f"{row['status']:16s} {row['dataset']:10s} "
          f"{', '.join(files) if files else row.get('error', '')}")


## Readouts — what this session produced

Three quick looks, one per arm. These only READ the per-dataset CSVs (guarded — a
dataset that has not run yet is reported, not crashed on). Cross-model scoring happens
later, once several sibling notebooks have run, by globbing `results/*/*.csv`.


In [ ]:
# RUL datasets — full-fleet snapshot of THIS backbone vs every baseline, per dataset.
import pandas as pd
from pathlib import Path

shown = 0
for ds in DATASETS:
    path = Path(RESULTS) / ds / f'{ds}_{MODEL_TAG}_results_v2.csv'
    if not path.exists():
        continue
    df = pd.read_csv(path)
    full = df[df.n_units == df.n_units.max()]
    mean = (full.groupby(['model', 'loss'])[['nasa_clipped', 'rmse_clipped']]
                .mean().reset_index())
    best = mean.loc[mean.groupby('model').nasa_clipped.idxmin()].sort_values('nasa_clipped')
    print(f'\n=== {ds} (n_units={int(df.n_units.max())}, seed-mean, best loss arm) ===')
    print(best.to_string(index=False))
    shown += 1
if not shown:
    print('no RUL results yet — run the campaign cell first')


In [ ]:
# Censored fleets — the alarm chapter. The win-rule is direction-aware here: alarm
# metrics are SKILL scores (higher is better), never tabled against RUL errors (§54).
from pathlib import Path
from src import scoring, plots

for ds in ('MetroPT-3', 'Backblaze'):
    d = Path(RESULTS) / ds
    csvs = sorted(d.glob('*alarm_results.csv'))
    if not csvs:
        print(f'{ds}: no alarm results yet')
        continue
    for metric in ('alarm_ap', 'alarm_auroc'):
        table = scoring.success_map(str(d / '*alarm_results.csv'), config,
                                    metric=metric, secondary_metric='alarm_recall')
        print(f'\n=== {ds} · {metric} ===')
        for row in table:
            print(f"  n={row['n_units']:>6} {row['model']:24s} {row['verdict']:6s} "
                  f"margin={row['margin']:+.3f} p={row['p']:.3f} vs {row['best_baseline']}")
    for p in csvs:
        plots.plot_alarm_scaling(p, d / 'figures',
                                 prefix=p.stem.replace('alarm_results', ''), show=True)


In [ ]:
# Hydraulic — the RQ-F chapter: does the frozen embedding separate adjust-vs-replace
# with few labels better than catch22 indicators / raw window stats?
from pathlib import Path
from src import plots
from src.evaluate import load_results

d = Path(RESULTS) / 'Hydraulic'
tax = sorted(d.glob('*taxonomy.csv'))
if not tax:
    print('Hydraulic: no taxonomy results yet')
for p in tax:
    rows = load_results(p)
    by = {}
    for r in rows:
        by.setdefault((r['feature_source'], int(r['shots'])), []).append(float(r['macro_f1']))
    print(f'=== {p.name} (macro-F1, seed-mean) ===')
    for (source, shots) in sorted(by):
        vals = by[(source, shots)]
        print(f'  {source:14s} shots={shots:>4}  {sum(vals)/len(vals):.3f}  (n={len(vals)})')
    plots.plot_taxonomy(p, d / 'figures', prefix=p.stem.replace('taxonomy', ''), show=True)


## Optional — the RQ-D / RQ-G factor probes for this backbone

RQ-D (raw vs. hand-crafted indicators, on XJTU's 25.6 kHz waveforms) and RQ-G (sampling
stride / aggregation richness, on N-CMAPSS DS02). Both need their datasets downloaded.
Baselines run **once, in the chronos session** (the §47/§48 pattern — the other four
sessions run models-only so no baseline row is duplicated); every session writes its own
`probe_<factor>_<tag>.csv`, so two sessions never append to one Drive CSV.


In [ ]:
RUN_PROBES = False   # set True to run the RQ-D / RQ-G probes for this backbone

if RUN_PROBES:
    from src.probes import run_factor_probe

    # RQ-D — do TSFMs make hand-crafted condition indicators obsolete?
    xjtu = config.replace(dataset='XJTU-SY', sensor_columns=None,
                          results_dir=f'{RESULTS}/XJTU-SY',
                          **DEFAULT_DATASET_OVERRIDES['XJTU-SY'])
    run_factor_probe(xjtu, 'feature_mode', levels={
        'indicators':   {'xjtu_feature_mode': 'indicators'},
        'raw_decimate': {'xjtu_feature_mode': 'raw', 'xjtu_raw_reduce': 'decimate'},
        'raw_segrms':   {'xjtu_feature_mode': 'raw', 'xjtu_raw_reduce': 'segment_rms'},
        'raw_plus':     {'xjtu_feature_mode': 'raw+indicators'},
    }, models=[MODEL], baselines=[], device=device,
       out_csv=f'{RESULTS}/XJTU-SY/probe_feature_mode_{TAG}.csv')

    # RQ-G — how finely must you sample, and how should sub-cycle data be aggregated?
    ds02 = config.replace(dataset='DS02', sensor_columns=None,
                          results_dir=f'{RESULTS}/DS02')
    run_factor_probe(ds02, 'aggregation', levels={
        '1hz_meanstd': {'ncmapss_agg_stride': 1,  'ncmapss_agg_stats': 'mean_std'},
        'stride10':    {'ncmapss_agg_stride': 10, 'ncmapss_agg_stats': 'mean_std'},
        'stride60':    {'ncmapss_agg_stride': 60, 'ncmapss_agg_stats': 'mean_std'},
        'rich':        {'ncmapss_agg_stride': 1,  'ncmapss_agg_stats': 'mean_std_minmax_slope'},
    }, models=[MODEL], baselines=[], device=device,
       out_csv=f'{RESULTS}/DS02/probe_aggregation_{TAG}.csv')
else:
    print('set RUN_PROBES = True to run the RQ-D / RQ-G factor probes for this backbone')
